Create CESNET 6 hours code 

In [ ]:
import pandas as pd
import numpy as np
import os
import glob

def aggregate_6_hours(input_dir, output_dir):
    """
    Agrega dados de vazão de 1 em 1 hora para 6 em 6 horas
    """
    
    # Criar diretório de saída se não existir
    os.makedirs(output_dir, exist_ok=True)
    
    # Encontrar todos os arquivos CSV no diretório de entrada
    csv_files = glob.glob(os.path.join(input_dir, "*throughput.csv"))
    
    print(f"Encontrados {len(csv_files)} arquivos para processar")
    
    for file_path in csv_files:
        try:
            df = pd.read_csv(file_path)
            
            required_columns = ['time', 'throughput_bps']
            if not all(col in df.columns for col in required_columns):
                print(f"Arquivo {os.path.basename(file_path)} não possui as colunas necessárias. Pulando...")
                continue
            
            df['time'] = pd.to_datetime(df['time'])
            
            df = df.sort_values('time')
            
            # Agrupar por períodos de 6 horas e calcular a média
            # Usando resample com frequência de 6 horas
            df_agg = df.set_index('time').resample('6H').agg({
                'throughput_bps': 'mean'
            }).reset_index()
            
            # # Remover linhas com valores NaN (resultado de períodos sem dados)
            # df_agg = df_agg.dropna()
            
            # Formatar a coluna time para string (opcional, mantém formato consistente)
            df_agg['time'] = df_agg['time'].dt.strftime('%Y-%m-%d %H:%M:%S')
            
            output_file = os.path.join(output_dir, os.path.basename(file_path))
            df_agg.to_csv(output_file, index=False)
            
            print(f"Processado: {os.path.basename(file_path)} -> {len(df_agg)} registros")
            
        except Exception as e:
            print(f"Erro ao processar {file_path}: {str(e)}")


In [ ]:
input_directory = "../cesnet-institutions-throughput/institutions/agg_1_hour"  # Substitua pelo seu diretório
output_directory = "../cesnet-institutions-throughput/institutions/agg_6_hours"

aggregate_6_hours(input_directory, output_directory)

print("Processamento concluído!")

In [ ]:
import os
import pandas as pd
import re
from pathlib import Path
import numpy as np

def agregar_datasets_por_link(diretorio_origem, diretorio_destino, protocolo):
    """
    Agrega todos os arquivos CSV de um diretório por link, agrupando medições de 6 em 6h.
    """
    
    # Criar diretório de destino se não existir
    Path(diretorio_destino).mkdir(parents=True, exist_ok=True)
    
    # Dicionário para armazenar dados agrupados por link
    dados_por_link = {}
    
    # Padrão regex para extrair informações do nome do arquivo
    padrao_nome = re.compile(rf'({protocolo})\s+esmond\s+data\s+([a-z]+-[a-z]+)\s+(\d{{2}}-\d{{2}}-\d{{4}})\.csv', re.IGNORECASE)
    
    # Contadores para estatísticas
    total_arquivos = 0
    arquivos_processados = 0
    arquivos_descartados = 0
    
    print(f"Processando arquivos em: {diretorio_origem}")
    print(f"Protocolo: {protocolo}")
    print("-" * 50)
    
    # Percorrer todos os arquivos CSV no diretório
    for arquivo in os.listdir(diretorio_origem):
        if arquivo.endswith('.csv'):
            total_arquivos += 1
            caminho_arquivo = os.path.join(diretorio_origem, arquivo)
            
            # Tentar extrair informações do nome do arquivo
            match = padrao_nome.search(arquivo)
            if not match:
                print(f"Arquivo com nome fora do padrão: {arquivo}")
                continue
            
            protocolo_arquivo, link, data = match.groups()
            
            try:
                df = pd.read_csv(caminho_arquivo)
                
                if len(df) < 2:
                    arquivos_descartados += 1
                    print(f"Descartado (poucos dados): {arquivo} - {len(df)} linhas")
                    continue
                
                # Adicionar colunas de link e data para referência
                df['link'] = link
                df['data_arquivo'] = data
                df['protocolo'] = protocolo_arquivo
                
                # Converter timestamp se existir
                if 'Timestamp' in df.columns:
                    df['Timestamp'] = pd.to_datetime(df['Timestamp'], unit='s')
                
                # Converter coluna Data para datetime se existir
                if 'Data' in df.columns:
                    df['Data'] = pd.to_datetime(df['Data'])
                
                # Agrupar por link
                if link not in dados_por_link:
                    dados_por_link[link] = []
                
                dados_por_link[link].append(df)
                arquivos_processados += 1
                
                print(f"Processado: {arquivo} - Link: {link} - Linhas: {len(df)}")
                
            except Exception as e:
                print(f"Erro ao processar {arquivo}: {e}")
                arquivos_descartados += 1
    
    print("\n" + "=" * 50)
    print(f"Estatísticas:")
    print(f"Total de arquivos CSV: {total_arquivos}")
    print(f"Arquivos processados: {arquivos_processados}")
    print(f"Arquivos descartados: {arquivos_descartados}")
    print("=" * 50 + "\n")
    
    # Processar e salvar dados agrupados por link
    for link, dataframes in dados_por_link.items():
        if dataframes:
            # Concatenar todos os dataframes do mesmo link
            df_agregado = pd.concat(dataframes, ignore_index=True)
            
            # Ordenar por timestamp se existir
            if 'Timestamp' in df_agregado.columns:
                df_agregado = df_agregado.sort_values('Timestamp')
            
            # Criar todos os intervalos de 6 horas para o período completo
            if 'Data' in df_agregado.columns:
                # Encontrar data mínima e máxima
                min_date = df_agregado['Data'].min().floor('D')
                max_date = df_agregado['Data'].max().ceil('D')
                
                # Criar todos os intervalos de 6 horas
                all_intervals = pd.date_range(
                    start=min_date, 
                    end=max_date, 
                    freq='6H'
                )
                
                # Criar DataFrame com todos os intervalos
                df_all_intervals = pd.DataFrame({'Data': all_intervals})
                
                # Arredondar as datas originais para os intervalos de 6 horas mais próximos
                df_agregado['Data_intervalo'] = df_agregado['Data'].dt.floor('6H')
                
                # Agrupar por intervalo e calcular a média da vazão
                df_grouped = df_agregado.groupby('Data_intervalo')['Vazao'].mean().reset_index()
                df_grouped.rename(columns={'Data_intervalo': 'Data'}, inplace=True)
                
                # Juntar com todos os intervalos para garantir que todos estejam presentes
                df_final = pd.merge(df_all_intervals, df_grouped, on='Data', how='left')
                
                # Formatar a coluna Data para o formato desejado
                df_final['Data'] = df_final['Data'].dt.strftime('%Y-%m-%d %H:%M:%S')
            else:
                # Se não houver coluna Data, usar o método anterior
                df_agregado['grupo_6h'] = df_agregado['Timestamp'].dt.floor('6H')
                colunas_agregacao = [col for col in df_agregado.columns 
                                   if col not in ['Timestamp', 'grupo_6h', 'link', 'data_arquivo', 'protocolo']]
                df_final = df_agregado.groupby('grupo_6h')[colunas_agregacao].mean().reset_index()
                df_final.rename(columns={'grupo_6h': 'Data'}, inplace=True)
                df_final['Data'] = df_final['Data'].dt.strftime('%Y-%m-%d %H:%M:%S')
            
            # Manter apenas as colunas Data e Vazao
            df_final = df_final[['Data', 'Vazao']]
            
            # Nome do arquivo de saída
            nome_arquivo_saida = f"{protocolo}_esmond_data_{link}_2024-2025.csv"
            caminho_saida = os.path.join(diretorio_destino, nome_arquivo_saida)
            
            # Salvar arquivo agregado
            df_final.to_csv(caminho_saida, index=False)
            
            print(f"Salvo: {nome_arquivo_saida} - {len(df_final)} linhas agregadas")
    
    print(f"\nProcessamento concluído! Arquivos salvos em: {diretorio_destino}")

In [ ]:
# Diretórios de origem e destino
protocolo = "bbr"  # ou "cubic"
diretorio_origem = f"../vazao-2024-2025/{protocolo}"
diretorio_destino = f"../vazao-2024-2025-agg/{protocolo}"

# Executar a agregação
agregar_datasets_por_link(diretorio_origem, diretorio_destino, protocolo)

In [ ]:
# Diretórios de origem e destino
protocolo2 = "cubic"  # ou "cubic"
diretorio_origem2 = f"../vazao-2024-2025/{protocolo2}"
diretorio_destino2 = f"../vazao-2024-2025-agg/{protocolo2}"

# Executar a agregação
agregar_datasets_por_link(diretorio_origem2, diretorio_destino2, protocolo2)

In [ ]:
import os
import pandas as pd
import numpy as np
from pathlib import Path
import re

def gerar_relatorio_detalhado(diretorio_agg):
    """
    Gera um relatório detalhado sobre os datasets agregados.
    
    Args:
        diretorio_agg (str): Caminho do diretório com datasets agregados
    """
    
    # Dicionários para armazenar informações
    relatorio = {
        'protocolos': {},
        'estatisticas_gerais': {
            'total_arquivos': 0,
            'total_links': 0,
            'protocolos_encontrados': []
        }
    }
    
    print("=" * 80)
    print("RELATÓRIO DETALHADO - DATASETS DE VAZÃO 2024-2025")
    print("=" * 80)
    
    # Verificar se o diretório existe
    if not os.path.exists(diretorio_agg):
        print(f"Erro: Diretório {diretorio_agg} não encontrado!")
        return
    
    # Percorrer todos os arquivos CSV no diretório
    for arquivo in os.listdir(diretorio_agg):
        if arquivo.endswith('.csv'):
            caminho_arquivo = os.path.join(diretorio_agg, arquivo)
            
            # Extrair informações do nome do arquivo
            match = re.search(r'^(bbr|cubic)_esmond_data_([a-z]+-[a-z]+)_2024-2025\.csv$', arquivo)
            if not match:
                print(f"Arquivo com nome fora do padrão: {arquivo}")
                continue
            
            protocolo, link = match.groups()
            relatorio['estatisticas_gerais']['total_arquivos'] += 1
            
            # Inicializar estrutura para o protocolo se não existir
            if protocolo not in relatorio['protocolos']:
                relatorio['protocolos'][protocolo] = {
                    'quantidade_datasets': 0,
                    'links': [],
                    'dados_faltantes': [],
                    'intervalos_tempo': [],
                    'max_buracos': [],
                    'media_buracos': []
                }
                relatorio['estatisticas_gerais']['protocolos_encontrados'].append(protocolo)
            
            # Adicionar informações do arquivo atual
            relatorio['protocolos'][protocolo]['quantidade_datasets'] += 1
            relatorio['protocolos'][protocolo]['links'].append(link)
            
            try:
                # Ler o arquivo CSV
                df = pd.read_csv(caminho_arquivo)
                
                # Converter coluna Data para datetime
                df['Data'] = pd.to_datetime(df['Data'])
                
                # 1. Quantidade de dados faltantes
                dados_faltantes = df['Vazao'].isna().sum()
                relatorio['protocolos'][protocolo]['dados_faltantes'].append(dados_faltantes)
                
                # 2. Intervalo de tempo do dataset
                data_min = df['Data'].min()
                data_max = df['Data'].max()
                intervalo_tempo = (data_max - data_min).days
                relatorio['protocolos'][protocolo]['intervalos_tempo'].append(intervalo_tempo)
                
                # 3. Maior buraco de dados faltantes (sequência consecutiva de NaN)
                max_buraco = 0
                current_buraco = 0
                
                for is_na in df['Vazao'].isna():
                    if is_na:
                        current_buraco += 1
                        max_buraco = max(max_buraco, current_buraco)
                    else:
                        current_buraco = 0
                
                relatorio['protocolos'][protocolo]['max_buracos'].append(max_buraco)
                
                # 4. Média dos buracos (tamanho médio das sequências de NaN)
                buracos = []
                current_buraco = 0
                
                for is_na in df['Vazao'].isna():
                    if is_na:
                        current_buraco += 1
                    elif current_buraco > 0:
                        buracos.append(current_buraco)
                        current_buraco = 0
                
                # Adicionar o último buraco se existir
                if current_buraco > 0:
                    buracos.append(current_buraco)
                
                media_buracos = np.mean(buracos) if buracos else 0
                relatorio['protocolos'][protocolo]['media_buracos'].append(media_buracos)
                
                print(f"Processado: {arquivo}")
                print(f"  - Dados faltantes: {dados_faltantes}")
                print(f"  - Intervalo tempo: {intervalo_tempo} dias")
                print(f"  - Maior buraco: {max_buraco} intervalos")
                print(f"  - Média buracos: {media_buracos:.2f} intervalos")
                print("-" * 40)
                
            except Exception as e:
                print(f"Erro ao processar {arquivo}: {e}")
    
    # Calcular estatísticas gerais
    relatorio['estatisticas_gerais']['total_links'] = sum(
        len(proto_info['links']) 
        for proto_info in relatorio['protocolos'].values()
    )
    
    # Gerar relatório final
    print("\n" + "=" * 80)
    print("RELATÓRIO FINAL")
    print("=" * 80)
    
    for protocolo, info in relatorio['protocolos'].items():
        print(f"\nPROTOCOLO: {protocolo.upper()}")
        print("-" * 40)
        
        # Quantidade de datasets
        print(f"Quantidade de datasets: {info['quantidade_datasets']}")
        
        # Links únicos
        links_unicos = sorted(set(info['links']))
        print(f"Links encontrados ({len(links_unicos)}): {', '.join(links_unicos)}")
        
        # Dados faltantes
        if info['dados_faltantes']:
            print(f"Dados faltantes - Total: {sum(info['dados_faltantes'])}")
            print(f"Dados faltantes - Média por dataset: {np.mean(info['dados_faltantes']):.2f}")
            print(f"Dados faltantes - Máximo: {max(info['dados_faltantes'])}")
            print(f"Dados faltantes - Mínimo: {min(info['dados_faltantes'])}")
        
        # Intervalos de tempo
        if info['intervalos_tempo']:
            print(f"Intervalo tempo - Média: {np.mean(info['intervalos_tempo']):.2f} dias")
            print(f"Intervalo tempo - Máximo: {max(info['intervalos_tempo'])} dias")
            print(f"Intervalo tempo - Mínimo: {min(info['intervalos_tempo'])} dias")
        
        # Maiores buracos
        if info['max_buracos']:
            print(f"Maior buraco - Média: {np.mean(info['max_buracos']):.2f} intervalos")
            print(f"Maior buraco - Máximo: {max(info['max_buracos'])} intervalos")
            print(f"Maior buraco - Mínimo: {min(info['max_buracos'])} intervalos")
        
        # Médias dos buracos
        if info['media_buracos']:
            print(f"Média buracos - Geral: {np.mean(info['media_buracos']):.2f} intervalos")
            print(f"Média buracos - Máxima: {max(info['media_buracos']):.2f} intervalos")
            print(f"Média buracos - Mínima: {min(info['media_buracos']):.2f} intervalos")
    
    # Estatísticas gerais
    print(f"\nESTATÍSTICAS GERAIS")
    print("-" * 40)
    print(f"Total de arquivos processados: {relatorio['estatisticas_gerais']['total_arquivos']}")
    print(f"Total de links únicos: {relatorio['estatisticas_gerais']['total_links']}")
    print(f"Protocolos encontrados: {', '.join(relatorio['estatisticas_gerais']['protocolos_encontrados'])}")
    
    # Salvar relatório em arquivo
    salvar_relatorio_txt(relatorio, diretorio_agg)
    
    return relatorio

def salvar_relatorio_txt(relatorio, diretorio_agg):
    """Salva o relatório em um arquivo texto."""
    caminho_relatorio = os.path.join(diretorio_agg, "relatorio_detalhado.txt")
    
    with open(caminho_relatorio, 'w', encoding='utf-8') as f:
        f.write("=" * 80 + "\n")
        f.write("RELATÓRIO DETALHADO - DATASETS DE VAZÃO 2024-2025\n")
        f.write("=" * 80 + "\n\n")
        
        for protocolo, info in relatorio['protocolos'].items():
            f.write(f"PROTOCOLO: {protocolo.upper()}\n")
            f.write("-" * 40 + "\n")
            
            f.write(f"Quantidade de datasets: {info['quantidade_datasets']}\n")
            
            links_unicos = sorted(set(info['links']))
            f.write(f"Links encontrados ({len(links_unicos)}): {', '.join(links_unicos)}\n")
            
            if info['dados_faltantes']:
                f.write(f"Dados faltantes - Total: {sum(info['dados_faltantes'])}\n")
                f.write(f"Dados faltantes - Média por dataset: {np.mean(info['dados_faltantes']):.2f}\n")
                f.write(f"Dados faltantes - Máximo: {max(info['dados_faltantes'])}\n")
                f.write(f"Dados faltantes - Mínimo: {min(info['dados_faltantes'])}\n")
            
            if info['intervalos_tempo']:
                f.write(f"Intervalo tempo - Média: {np.mean(info['intervalos_tempo']):.2f} dias\n")
                f.write(f"Intervalo tempo - Máximo: {max(info['intervalos_tempo'])} dias\n")
                f.write(f"Intervalo tempo - Mínimo: {min(info['intervalos_tempo'])} dias\n")
            
            if info['max_buracos']:
                f.write(f"Maior buraco - Média: {np.mean(info['max_buracos']):.2f} intervalos\n")
                f.write(f"Maior buraco - Máximo: {max(info['max_buracos'])} intervalos\n")
                f.write(f"Maior buraco - Mínimo: {min(info['max_buracos'])} intervalos\n")
            
            if info['media_buracos']:
                f.write(f"Média buracos - Geral: {np.mean(info['media_buracos']):.2f} intervalos\n")
                f.write(f"Média buracos - Máxima: {max(info['media_buracos']):.2f} intervalos\n")
                f.write(f"Média buracos - Mínima: {min(info['media_buracos']):.2f} intervalos\n")
            
            f.write("\n")
        
        f.write("ESTATÍSTICAS GERAIS\n")
        f.write("-" * 40 + "\n")
        f.write(f"Total de arquivos processados: {relatorio['estatisticas_gerais']['total_arquivos']}\n")
        f.write(f"Total de links únicos: {relatorio['estatisticas_gerais']['total_links']}\n")
        f.write(f"Protocolos encontrados: {', '.join(relatorio['estatisticas_gerais']['protocolos_encontrados'])}\n")
    
    print(f"\nRelatório salvo em: {caminho_relatorio}")


# Se quiser gerar relatório para um protocolo específico
relatorio_bbr = gerar_relatorio_detalhado("../vazao-2024-2025-agg/bbr")
relatorio_cubic = gerar_relatorio_detalhado("../vazao-2024-2025-agg/cubic")

# ANÁLISE de Intervalos mais Longos

In [1]:
import os
import pandas as pd
import numpy as np
from pathlib import Path
import re
from datetime import timedelta

def analisar_intervalo_longo_com_falhas(diretorio_agg):
    """
    Analisa os datasets agregados focando no intervalo mais longo com até 1% de falhas.
    """
    
    resultados = []
    
    print("=" * 100)
    print("ANÁLISE DE INTERVALO MAIS LONGO COM ATÉ 1% DE FALHAS")
    print("=" * 100)
    
    # Verificar se o diretório existe
    if not os.path.exists(diretorio_agg):
        print(f"Erro: Diretório {diretorio_agg} não encontrado!")
        return []
    
    # Percorrer todos os arquivos CSV no diretório
    for arquivo in os.listdir(diretorio_agg):
        if arquivo.endswith('.csv'):
            caminho_arquivo = os.path.join(diretorio_agg, arquivo)
            
            # Extrair informações do nome do arquivo
            match = re.search(r'^(bbr|cubic)_esmond_data_([a-z]+-[a-z]+)_2024-2025\.csv$', arquivo)
            if not match:
                print(f"Arquivo com nome fora do padrão: {arquivo}")
                continue
            
            protocolo, link = match.groups()
            
            try:
                # Ler o arquivo CSV
                df = pd.read_csv(caminho_arquivo)
                
                # Converter coluna Data para datetime
                df['Data'] = pd.to_datetime(df['Data'])
                
                # Ordenar por data
                df = df.sort_values('Data').reset_index(drop=True)
                
                # Encontrar o intervalo mais longo com até 1% de falhas
                intervalo_info = encontrar_intervalo_mais_longo(df, max_falhas_percent=0.01)
                
                # Adicionar informações ao resultado
                resultado = {
                    'protocolo': protocolo,
                    'link': link,
                    'arquivo': arquivo,
                    'intervalo_mais_longo_dias': intervalo_info['dias'],
                    'amostras_intervalo': intervalo_info['amostras'],
                    'falhas_intervalo': intervalo_info['falhas'],
                    'falhas_percent': intervalo_info['falhas_percent'],
                    'data_inicio': intervalo_info['inicio'],
                    'data_fim': intervalo_info['fim'],
                    'total_amostras': len(df),
                    'total_falhas': df['Vazao'].isna().sum(),
                    'total_falhas_percent': (df['Vazao'].isna().sum() / len(df)) * 100
                }
                
                resultados.append(resultado)
                
                print(f"{arquivo}:")
                print(f"  - Intervalo mais longo: {intervalo_info['dias']:.1f} dias")
                print(f"  - Amostras no intervalo: {intervalo_info['amostras']}")
                print(f"  - Falhas no intervalo: {intervalo_info['falhas']} ({intervalo_info['falhas_percent']:.2f}%)")
                print(f"  - Período: {intervalo_info['inicio'].date()} a {intervalo_info['fim'].date()}")
                print("-" * 60)
                
            except Exception as e:
                print(f"Erro ao processar {arquivo}: {e}")
    
    return resultados

def encontrar_intervalo_mais_longo(df, max_falhas_percent=0.01):
    """
    Encontra o intervalo contínuo mais longo com até max_falhas_percent de falhas.
    
    Args:
        df: DataFrame com colunas 'Data' e 'Vazao'
        max_falhas_percent: Percentual máximo de falhas permitido (0.01 = 1%)
    
    Returns:
        Dicionário com informações do intervalo
    """
    # Verificar se há dados
    if len(df) == 0:
        return {'dias': 0, 'amostras': 0, 'falhas': 0, 'falhas_percent': 0, 'inicio': None, 'fim': None}
    
    # Ordenar por data
    df = df.sort_values('Data').reset_index(drop=True)
    
    max_intervalo_dias = 0
    melhor_inicio = 0
    melhor_fim = 0
    
    # Usar sliding window para encontrar o melhor intervalo
    left = 0
    falhas_atual = 0
    total_amostras = 0
    
    for right in range(len(df)):
        # Contar falha se presente
        if pd.isna(df.loc[right, 'Vazao']):
            falhas_atual += 1
        
        total_amostras = right - left + 1
        falhas_percent = falhas_atual / total_amostras if total_amostras > 0 else 0
        
        # Mover left até que as falhas estejam dentro do limite
        while falhas_percent > max_falhas_percent and left <= right:
            if pd.isna(df.loc[left, 'Vazao']):
                falhas_atual -= 1
            left += 1
            total_amostras = right - left + 1
            falhas_percent = falhas_atual / total_amostras if total_amostras > 0 else 0
        
        # Calcular duração do intervalo atual
        if left <= right:
            data_inicio = df.loc[left, 'Data']
            data_fim = df.loc[right, 'Data']
            intervalo_dias = (data_fim - data_inicio).total_seconds() / (24 * 3600)
            
            # Atualizar melhor intervalo se necessário
            if intervalo_dias > max_intervalo_dias:
                max_intervalo_dias = intervalo_dias
                melhor_inicio = left
                melhor_fim = right
    
    # Calcular estatísticas do melhor intervalo
    if melhor_inicio <= melhor_fim:
        inicio_data = df.loc[melhor_inicio, 'Data']
        fim_data = df.loc[melhor_fim, 'Data']
        amostras_intervalo = melhor_fim - melhor_inicio + 1
        
        # Contar falhas no intervalo
        falhas_intervalo = df.loc[melhor_inicio:melhor_fim, 'Vazao'].isna().sum()
        falhas_percent_intervalo = (falhas_intervalo / amostras_intervalo) * 100
        
        return {
            'dias': max_intervalo_dias,
            'amostras': amostras_intervalo,
            'falhas': falhas_intervalo,
            'falhas_percent': falhas_percent_intervalo,
            'inicio': inicio_data,
            'fim': fim_data
        }
    else:
        return {'dias': 0, 'amostras': 0, 'falhas': 0, 'falhas_percent': 0, 'inicio': None, 'fim': None}

def gerar_relatorio_completo(diretorio_agg):
    """
    Gera relatório completo com análise de intervalo mais longo.
    """
    
    # Analisar todos os datasets
    resultados = analisar_intervalo_longo_com_falhas(diretorio_agg)
    
    if not resultados:
        print("Nenhum resultado encontrado!")
        return
    
    # Ordenar por intervalo mais longo (decrescente)
    resultados_ordenados = sorted(resultados, key=lambda x: x['intervalo_mais_longo_dias'], reverse=True)
    
    # Calcular estatísticas gerais
    estatisticas = {
        'total_datasets': len(resultados),
        'protocolos': {},
        'intervalo_medio': np.mean([r['intervalo_mais_longo_dias'] for r in resultados]),
        'intervalo_maximo': max([r['intervalo_mais_longo_dias'] for r in resultados]),
        'amostras_medias': np.mean([r['amostras_intervalo'] for r in resultados])
    }
    
    # Agrupar por protocolo
    for resultado in resultados:
        protocolo = resultado['protocolo']
        if protocolo not in estatisticas['protocolos']:
            estatisticas['protocolos'][protocolo] = {
                'count': 0,
                'intervalos': [],
                'amostras': []
            }
        estatisticas['protocolos'][protocolo]['count'] += 1
        estatisticas['protocolos'][protocolo]['intervalos'].append(resultado['intervalo_mais_longo_dias'])
        estatisticas['protocolos'][protocolo]['amostras'].append(resultado['amostras_intervalo'])
    
    # Gerar relatório
    print("\n" + "=" * 100)
    print("RELATÓRIO COMPLETO - ORDENADO POR INTERVALO MAIS LONGO")
    print("=" * 100)
    
    print(f"\nESTATÍSTICAS GERAIS:")
    print(f"Total de datasets analisados: {estatisticas['total_datasets']}")
    print(f"Intervalo médio mais longo: {estatisticas['intervalo_medio']:.1f} dias")
    print(f"Intervalo máximo mais longo: {estatisticas['intervalo_maximo']:.1f} dias")
    print(f"Média de amostras por intervalo: {estatisticas['amostras_medias']:.0f}")
    
    for protocolo, info in estatisticas['protocolos'].items():
        print(f"\n{protocolo.upper()}: {info['count']} datasets")
        print(f"  Intervalo médio: {np.mean(info['intervalos']):.1f} dias")
        print(f"  Amostras médias: {np.mean(info['amostras']):.0f}")
    
    print(f"\n{'='*100}")
    print("RANKING POR INTERVALO MAIS LONGO:")
    print(f"{'Pos':<4} {'Protocolo':<8} {'Link':<10} {'Dias':<8} {'Amostras':<10} {'Falhas':<8} {'Período'}")
    print(f"{'-'*100}")
    
    for i, resultado in enumerate(resultados_ordenados, 1):
        periodo = f"{resultado['data_inicio'].date()} a {resultado['data_fim'].date()}"
        print(f"{i:<4} {resultado['protocolo']:<8} {resultado['link']:<10} "
              f"{resultado['intervalo_mais_longo_dias']:<8.1f} "
              f"{resultado['amostras_intervalo']:<10} "
              f"{resultado['falhas_intervalo']:<8} "
              f"{periodo}")
    
    # Salvar relatório detalhado
    salvar_relatorio_detalhado(resultados_ordenados, estatisticas, diretorio_agg)
    
    return resultados_ordenados, estatisticas

def salvar_relatorio_detalhado(resultados, estatisticas, diretorio_agg):
    """Salva o relatório completo em arquivo CSV e TXT."""
    
    # Salvar CSV com todos os resultados
    df_resultados = pd.DataFrame(resultados)
    caminho_csv = os.path.join(diretorio_agg, "analise_intervalos_longo.csv")
    df_resultados.to_csv(caminho_csv, index=False, encoding='utf-8')
    
    # Salvar relatório TXT
    caminho_txt = os.path.join(diretorio_agg, "relatorio_intervalos_longo.txt")
    
    with open(caminho_txt, 'w', encoding='utf-8') as f:
        f.write("=" * 100 + "\n")
        f.write("RELATÓRIO DE INTERVALOS MAIS LONGOS COM ATÉ 1% DE FALHAS\n")
        f.write("=" * 100 + "\n\n")
        
        f.write("ESTATÍSTICAS GERAIS:\n")
        f.write(f"Total de datasets analisados: {estatisticas['total_datasets']}\n")
        f.write(f"Intervalo médio mais longo: {estatisticas['intervalo_medio']:.1f} dias\n")
        f.write(f"Intervalo máximo mais longo: {estatisticas['intervalo_maximo']:.1f} dias\n")
        f.write(f"Média de amostras por intervalo: {estatisticas['amostras_medias']:.0f}\n\n")
        
        for protocolo, info in estatisticas['protocolos'].items():
            f.write(f"{protocolo.upper()}: {info['count']} datasets\n")
            f.write(f"  Intervalo médio: {np.mean(info['intervalos']):.1f} dias\n")
            f.write(f"  Amostras médias: {np.mean(info['amostras']):.0f}\n")
        
        f.write("\n" + "=" * 100 + "\n")
        f.write("RANKING POR INTERVALO MAIS LONGO:\n")
        f.write("=" * 100 + "\n")
        f.write(f"{'Pos':<4} {'Protocolo':<8} {'Link':<10} {'Dias':<8} {'Amostras':<10} {'Falhas':<8} {'Período'}\n")
        f.write("-" * 100 + "\n")
        
        for i, resultado in enumerate(resultados, 1):
            periodo = f"{resultado['data_inicio'].date()} a {resultado['data_fim'].date()}"
            f.write(f"{i:<4} {resultado['protocolo']:<8} {resultado['link']:<10} "
                   f"{resultado['intervalo_mais_longo_dias']:<8.1f} "
                   f"{resultado['amostras_intervalo']:<10} "
                   f"{resultado['falhas_intervalo']:<8} "
                   f"{periodo}\n")
    
    print(f"\nRelatório salvo em:")
    print(f"CSV: {caminho_csv}")
    print(f"TXT: {caminho_txt}")

# Função principal para análise de diretório específico
def analisar_diretorio_protocolo(diretorio_protocolo):
    """Analisa um diretório específico de protocolo."""
    if os.path.exists(diretorio_protocolo):
        print(f"\nAnalisando: {diretorio_protocolo}")
        return gerar_relatorio_completo(diretorio_protocolo)
    else:
        print(f"Diretório não encontrado: {diretorio_protocolo}")
        return [], {}

# Exemplo de uso
if __name__ == "__main__":

    diretorio_base = "../data/vazao-escolhidos"
    resultados_ordenados, estatisticas = gerar_relatorio_completo(diretorio_base)

    # Ou analisar diretórios específicos
    # resultados_bbr, stats_bbr = analisar_diretorio_protocolo("../vazao-2024-2025-agg/bbr")
    # resultados_cubic, stats_cubic = analisar_diretorio_protocolo("../vazao-2024-2025-agg/cubic")

ANÁLISE DE INTERVALO MAIS LONGO COM ATÉ 1% DE FALHAS
bbr_esmond_data_ap-ac_2024-2025.csv:
  - Intervalo mais longo: 8.8 dias
  - Amostras no intervalo: 36
  - Falhas no intervalo: 0 (0.00%)
  - Período: 2024-08-03 a 2024-08-12
------------------------------------------------------------
bbr_esmond_data_ap-go_2024-2025.csv:
  - Intervalo mais longo: 14.2 dias
  - Amostras no intervalo: 58
  - Falhas no intervalo: 0 (0.00%)
  - Período: 2025-04-19 a 2025-05-03
------------------------------------------------------------
bbr_esmond_data_ba-ms_2024-2025.csv:
  - Intervalo mais longo: 9.8 dias
  - Amostras no intervalo: 40
  - Falhas no intervalo: 0 (0.00%)
  - Período: 2024-12-30 a 2025-01-09
------------------------------------------------------------
bbr_esmond_data_es-ac_2024-2025.csv:
  - Intervalo mais longo: 11.8 dias
  - Amostras no intervalo: 48
  - Falhas no intervalo: 0 (0.00%)
  - Período: 2024-12-19 a 2024-12-31
------------------------------------------------------------
bbr_e

# Gerar Datasets Intervalos Mais longos

In [1]:
import os
import pandas as pd
import numpy as np
from pathlib import Path
import re
from datetime import timedelta

def analisar_intervalo_longo_com_falhas(diretorio_agg, diretorio_saida_intervalos):
    """
    Analisa os datasets agregados focando no intervalo mais longo com até 1% de falhas.
    Salva os intervalos encontrados em arquivos separados.
    """
    
    resultados = []
    
    print("=" * 100)
    print("ANÁLISE DE INTERVALO MAIS LONGO COM ATÉ 1% DE FALHAS")
    print("=" * 100)
    
    # Verificar se os diretórios existem
    if not os.path.exists(diretorio_agg):
        print(f"Erro: Diretório {diretorio_agg} não encontrado!")
        return []
    
    # Criar diretório de saída se não existir
    os.makedirs(diretorio_saida_intervalos, exist_ok=True)
    
    # Percorrer todos os arquivos CSV no diretório
    for arquivo in os.listdir(diretorio_agg):
        if arquivo.endswith('.csv'):
            caminho_arquivo = os.path.join(diretorio_agg, arquivo)
            
            # Extrair informações do nome do arquivo
            match = re.search(r'^(bbr|cubic)_esmond_data_([a-z]+-[a-z]+)_2024-2025\.csv$', arquivo)
            if not match:
                print(f"Arquivo com nome fora do padrão: {arquivo}")
                continue
            
            protocolo, link = match.groups()
            
            try:
                # Ler o arquivo CSV
                df = pd.read_csv(caminho_arquivo)
                
                # Converter coluna Data para datetime
                df['Data'] = pd.to_datetime(df['Data'])
                
                # Ordenar por data
                df = df.sort_values('Data').reset_index(drop=True)
                
                # Encontrar o intervalo mais longo com até 1% de falhas
                intervalo_info = encontrar_intervalo_mais_longo(df, max_falhas_percent=0.01)
                
                # Adicionar informações ao resultado
                resultado = {
                    'protocolo': protocolo,
                    'link': link,
                    'arquivo': arquivo,
                    'intervalo_mais_longo_dias': intervalo_info['dias'],
                    'amostras_intervalo': intervalo_info['amostras'],
                    'falhas_intervalo': intervalo_info['falhas'],
                    'falhas_percent': intervalo_info['falhas_percent'],
                    'data_inicio': intervalo_info['inicio'],
                    'data_fim': intervalo_info['fim'],
                    'total_amostras': len(df),
                    'total_falhas': df['Vazao'].isna().sum(),
                    'total_falhas_percent': (df['Vazao'].isna().sum() / len(df)) * 100
                }
                
                resultados.append(resultado)
                
                # Salvar o intervalo encontrado
                if intervalo_info['amostras'] > 0:
                    df_intervalo = df.loc[intervalo_info['inicio_idx']:intervalo_info['fim_idx']].copy()
                    nome_arquivo_saida = f"{protocolo}_{link}_intervalo_longo.csv"
                    caminho_saida = os.path.join(diretorio_saida_intervalos, nome_arquivo_saida)
                    df_intervalo.to_csv(caminho_saida, index=False)
                    print(f"Intervalo salvo: {nome_arquivo_saida}")
                
                print(f"{arquivo}:")
                print(f"  - Intervalo mais longo: {intervalo_info['dias']:.1f} dias")
                print(f"  - Amostras no intervalo: {intervalo_info['amostras']}")
                print(f"  - Falhas no intervalo: {intervalo_info['falhas']} ({intervalo_info['falhas_percent']:.2f}%)")
                print(f"  - Período: {intervalo_info['inicio'].date()} a {intervalo_info['fim'].date()}")
                print("-" * 60)
                
            except Exception as e:
                print(f"Erro ao processar {arquivo}: {e}")
    
    return resultados

def encontrar_intervalo_mais_longo(df, max_falhas_percent=0.01):
    """
    Encontra o intervalo contínuo mais longo com até max_falhas_percent de falhas.
    
    Args:
        df: DataFrame com colunas 'Data' e 'Vazao'
        max_falhas_percent: Percentual máximo de falhas permitido (0.01 = 1%)
    
    Returns:
        Dicionário com informações do intervalo
    """
    # Verificar se há dados
    if len(df) == 0:
        return {'dias': 0, 'amostras': 0, 'falhas': 0, 'falhas_percent': 0, 'inicio': None, 'fim': None, 'inicio_idx': None, 'fim_idx': None}
    
    # Ordenar por data
    df = df.sort_values('Data').reset_index(drop=True)
    
    max_intervalo_dias = 0
    melhor_inicio = 0
    melhor_fim = 0
    
    # Usar sliding window para encontrar o melhor intervalo
    left = 0
    falhas_atual = 0
    total_amostras = 0
    
    for right in range(len(df)):
        # Contar falha se presente
        if pd.isna(df.loc[right, 'Vazao']):
            falhas_atual += 1
        
        total_amostras = right - left + 1
        falhas_percent = falhas_atual / total_amostras if total_amostras > 0 else 0
        
        # Mover left até que as falhas estejam dentro do limite
        while falhas_percent > max_falhas_percent and left <= right:
            if pd.isna(df.loc[left, 'Vazao']):
                falhas_atual -= 1
            left += 1
            total_amostras = right - left + 1
            falhas_percent = falhas_atual / total_amostras if total_amostras > 0 else 0
        
        # Calcular duração do intervalo atual
        if left <= right:
            data_inicio = df.loc[left, 'Data']
            data_fim = df.loc[right, 'Data']
            intervalo_dias = (data_fim - data_inicio).total_seconds() / (24 * 3600)
            
            # Atualizar melhor intervalo se necessário
            if intervalo_dias > max_intervalo_dias:
                max_intervalo_dias = intervalo_dias
                melhor_inicio = left
                melhor_fim = right
    
    # Calcular estatísticas do melhor intervalo
    if melhor_inicio <= melhor_fim:
        inicio_data = df.loc[melhor_inicio, 'Data']
        fim_data = df.loc[melhor_fim, 'Data']
        amostras_intervalo = melhor_fim - melhor_inicio + 1
        
        # Contar falhas no intervalo
        falhas_intervalo = df.loc[melhor_inicio:melhor_fim, 'Vazao'].isna().sum()
        falhas_percent_intervalo = (falhas_intervalo / amostras_intervalo) * 100
        
        return {
            'dias': max_intervalo_dias,
            'amostras': amostras_intervalo,
            'falhas': falhas_intervalo,
            'falhas_percent': falhas_percent_intervalo,
            'inicio': inicio_data,
            'fim': fim_data,
            'inicio_idx': melhor_inicio,
            'fim_idx': melhor_fim
        }
    else:
        return {'dias': 0, 'amostras': 0, 'falhas': 0, 'falhas_percent': 0, 'inicio': None, 'fim': None, 'inicio_idx': None, 'fim_idx': None}

def gerar_relatorio_completo(diretorio_agg, diretorio_saida_intervalos, caminho_relatorio):
    """
    Gera relatório completo com análise de intervalo mais longo.
    """
    
    # Analisar todos os datasets
    resultados = analisar_intervalo_longo_com_falhas(diretorio_agg, diretorio_saida_intervalos)
    
    if not resultados:
        print("Nenhum resultado encontrado!")
        return
    
    # Ordenar por intervalo mais longo (decrescente)
    resultados_ordenados = sorted(resultados, key=lambda x: x['intervalo_mais_longo_dias'], reverse=True)
    
    # Calcular estatísticas gerais
    estatisticas = {
        'total_datasets': len(resultados),
        'protocolos': {},
        'intervalo_medio': np.mean([r['intervalo_mais_longo_dias'] for r in resultados]),
        'intervalo_maximo': max([r['intervalo_mais_longo_dias'] for r in resultados]),
        'amostras_medias': np.mean([r['amostras_intervalo'] for r in resultados])
    }
    
    # Agrupar por protocolo
    for resultado in resultados:
        protocolo = resultado['protocolo']
        if protocolo not in estatisticas['protocolos']:
            estatisticas['protocolos'][protocolo] = {
                'count': 0,
                'intervalos': [],
                'amostras': []
            }
        estatisticas['protocolos'][protocolo]['count'] += 1
        estatisticas['protocolos'][protocolo]['intervalos'].append(resultado['intervalo_mais_longo_dias'])
        estatisticas['protocolos'][protocolo]['amostras'].append(resultado['amostras_intervalo'])
    
    # Gerar relatório
    print("\n" + "=" * 100)
    print("RELATÓRIO COMPLETO - ORDENADO POR INTERVALO MAIS LONGO")
    print("=" * 100)
    
    print(f"\nESTATÍSTICAS GERAIS:")
    print(f"Total de datasets analisados: {estatisticas['total_datasets']}")
    print(f"Intervalo médio mais longo: {estatisticas['intervalo_medio']:.1f} dias")
    print(f"Intervalo máximo mais longo: {estatisticas['intervalo_maximo']:.1f} dias")
    print(f"Média de amostras por intervalo: {estatisticas['amostras_medias']:.0f}")
    
    for protocolo, info in estatisticas['protocolos'].items():
        print(f"\n{protocolo.upper()}: {info['count']} datasets")
        print(f"  Intervalo médio: {np.mean(info['intervalos']):.1f} dias")
        print(f"  Amostras médias: {np.mean(info['amostras']):.0f}")
    
    print(f"\n{'='*100}")
    print("RANKING POR INTERVALO MAIS LONGO:")
    print(f"{'Pos':<4} {'Protocolo':<8} {'Link':<10} {'Dias':<8} {'Amostras':<10} {'Falhas':<8} {'Período'}")
    print(f"{'-'*100}")
    
    for i, resultado in enumerate(resultados_ordenados, 1):
        periodo = f"{resultado['data_inicio'].date()} a {resultado['data_fim'].date()}"
        print(f"{i:<4} {resultado['protocolo']:<8} {resultado['link']:<10} "
              f"{resultado['intervalo_mais_longo_dias']:<8.1f} "
              f"{resultado['amostras_intervalo']:<10} "
              f"{resultado['falhas_intervalo']:<8} "
              f"{periodo}")
    
    # Salvar relatório detalhado
    salvar_relatorio_detalhado(resultados_ordenados, estatisticas, caminho_relatorio)
    
    return resultados_ordenados, estatisticas

def salvar_relatorio_detalhado(resultados, estatisticas, caminho_relatorio):
    """Salva o relatório completo em arquivo CSV."""
    
    # Criar diretório se não existir
    os.makedirs(os.path.dirname(caminho_relatorio), exist_ok=True)
    
    # Salvar CSV com todos os resultados
    df_resultados = pd.DataFrame(resultados)
    df_resultados.to_csv(caminho_relatorio, index=False, encoding='utf-8')
    
    print(f"\nRelatório salvo em: {caminho_relatorio}")

In [2]:
diretorio_base = "../data/vazao-escolhidos"
diretorio_saida_intervalos = "../data/vazao-escolhidos-intervalos"
caminho_relatorio = "results/reports/geracao_intervalos_mais_longos.csv"

# Criar diretórios se não existirem
os.makedirs(diretorio_saida_intervalos, exist_ok=True)
os.makedirs(os.path.dirname(caminho_relatorio), exist_ok=True)

# Executar análise
resultados_ordenados, estatisticas = gerar_relatorio_completo(
    diretorio_base, 
    diretorio_saida_intervalos,
    caminho_relatorio
)

ANÁLISE DE INTERVALO MAIS LONGO COM ATÉ 1% DE FALHAS
Intervalo salvo: bbr_ap-ac_intervalo_longo.csv
bbr_esmond_data_ap-ac_2024-2025.csv:
  - Intervalo mais longo: 8.8 dias
  - Amostras no intervalo: 36
  - Falhas no intervalo: 0 (0.00%)
  - Período: 2024-08-03 a 2024-08-12
------------------------------------------------------------
Intervalo salvo: bbr_ap-go_intervalo_longo.csv
bbr_esmond_data_ap-go_2024-2025.csv:
  - Intervalo mais longo: 14.2 dias
  - Amostras no intervalo: 58
  - Falhas no intervalo: 0 (0.00%)
  - Período: 2025-04-19 a 2025-05-03
------------------------------------------------------------
Intervalo salvo: bbr_ba-ms_intervalo_longo.csv
bbr_esmond_data_ba-ms_2024-2025.csv:
  - Intervalo mais longo: 9.8 dias
  - Amostras no intervalo: 40
  - Falhas no intervalo: 0 (0.00%)
  - Período: 2024-12-30 a 2025-01-09
------------------------------------------------------------
Intervalo salvo: bbr_es-ac_intervalo_longo.csv
bbr_esmond_data_es-ac_2024-2025.csv:
  - Intervalo ma

# Seleção de melhores datasets - RNP

In [ ]:
import os
import pandas as pd
import numpy as np
import shutil
from pathlib import Path
import re

def selecionar_melhores_datasets(diretorio_origem, diretorio_destino, top_n=100):
    """
    Seleciona os melhores datasets baseado nos critérios:
    1. Quantidade de amostras do intervalo mais longo (mais importante)
    2. Completude total (menor porcentagem de dados faltantes)
    3. Quantidade total de amostras
    """
    
    # Criar diretório de destino se não existir
    Path(diretorio_destino).mkdir(parents=True, exist_ok=True)
    
    # Lista para armazenar scores de todos os datasets
    todos_datasets = []
    
    print("=" * 80)
    print("SELECIONANDOS OS MELHORES DATASETS")
    print("=" * 80)
    
    # Percorrer todos os arquivos CSV recursivamente
    for root, dirs, files in os.walk(diretorio_origem):
        for arquivo in files:
            if arquivo.endswith('.csv'):
                caminho_arquivo = os.path.join(root, arquivo)
                
                try:
                    # Ler o arquivo CSV
                    df = pd.read_csv(caminho_arquivo)
                    
                    # Pular arquivos muito pequenos
                    if len(df) < 10:
                        continue
                    
                    # Converter coluna Data para datetime se existir
                    if 'Data' in df.columns:
                        df['Data'] = pd.to_datetime(df['Data'])
                    
                    # Calcular métricas
                    total_amostras = len(df)
                    falhas_totais = df['Vazao'].isna().sum()
                    completude = 1 - (falhas_totais / total_amostras) if total_amostras > 0 else 0
                    
                    # Encontrar intervalo mais longo com até 5% de falhas
                    intervalo_info = encontrar_intervalo_mais_longo(df, max_falhas_percent=0.05)
                    
                    # Calcular score composto
                    score = calcular_score(
                        amostras_intervalo=intervalo_info['amostras'],
                        completude=completude,
                        total_amostras=total_amostras
                    )
                    
                    # Adicionar à lista
                    todos_datasets.append({
                        'caminho': caminho_arquivo,
                        'arquivo': arquivo,
                        'score': score,
                        'amostras_intervalo': intervalo_info['amostras'],
                        'completude': completude,
                        'total_amostras': total_amostras,
                        'falhas_totais': falhas_totais,
                        'intervalo_dias': intervalo_info['dias'],
                        'protocolo': extrair_protocolo(arquivo),
                        'link': extrair_link(arquivo)
                    })
                    
                    print(f"Processado: {arquivo} - Score: {score:.3f}")
                    
                except Exception as e:
                    print(f"Erro ao processar {arquivo}: {e}")
    
    # Ordenar por score (decrescente)
    todos_datasets_ordenados = sorted(todos_datasets, key=lambda x: x['score'], reverse=True)
    
    # Selecionar top N
    top_datasets = todos_datasets_ordenados[:top_n]
    
    # Copiar arquivos selecionados
    copiar_datasets_selecionados(top_datasets, diretorio_destino)
    
    # Gerar relatório
    gerar_relatorio_selecao(top_datasets, diretorio_destino)
    
    return top_datasets

def calcular_score(amostras_intervalo, completude, total_amostras):
    """
    Calcula score composto baseado nos critérios:
    1. Amostras do intervalo (peso 50%)
    2. Completude (peso 30%) 
    3. Total de amostras (peso 20%)
    """
    # Normalizar valores (assumindo máximos razoáveis)
    max_amostras_intervalo = 1000  # valor de referência
    max_total_amostras = 2000      # valor de referência
    
    # Componentes do score
    score_intervalo = min(1.0, amostras_intervalo / max_amostras_intervalo) * 0.5
    score_completude = completude * 0.3
    score_total = min(1.0, total_amostras / max_total_amostras) * 0.2
    
    return score_intervalo + score_completude + score_total

def encontrar_intervalo_mais_longo(df, max_falhas_percent=0.05):
    """
    Encontra o intervalo contínuo mais longo com até max_falhas_percent de falhas.
    """
    if len(df) == 0 or 'Vazao' not in df.columns:
        return {'dias': 0, 'amostras': 0, 'falhas': 0}
    
    # Ordenar por data se possível
    if 'Data' in df.columns:
        df = df.sort_values('Data').reset_index(drop=True)
    
    max_amostras = 0
    left = 0
    falhas_atual = 0
    
    for right in range(len(df)):
        # Contar falha se presente
        if pd.isna(df.loc[right, 'Vazao']):
            falhas_atual += 1
        
        total_window = right - left + 1
        falhas_percent = falhas_atual / total_window if total_window > 0 else 0
        
        # Mover left até que as falhas estejam dentro do limite
        while falhas_percent > max_falhas_percent and left <= right:
            if pd.isna(df.loc[left, 'Vazao']):
                falhas_atual -= 1
            left += 1
            total_window = right - left + 1
            falhas_percent = falhas_atual / total_window if total_window > 0 else 0
        
        # Atualizar máximo
        if left <= right and total_window > max_amostras:
            max_amostras = total_window
    
    # Calcular dias aproximados (assumindo medições a cada 6 horas)
    dias_aproximados = (max_amostras * 6) / 24
    
    return {
        'dias': dias_aproximados,
        'amostras': max_amostras,
        'falhas': falhas_atual
    }

def extrair_protocolo(nome_arquivo):
    """Extrai protocolo do nome do arquivo"""
    match = re.search(r'^(bbr|cubic)', nome_arquivo, re.IGNORECASE)
    return match.group(0).lower() if match else 'unknown'

def extrair_link(nome_arquivo):
    """Extrai link do nome do arquivo"""
    match = re.search(r'esmond_data_([a-z]+-[a-z]+)', nome_arquivo, re.IGNORECASE)
    return match.group(1) if match else 'unknown'

def copiar_datasets_selecionados(datasets, diretorio_destino):
    """Copia os datasets selecionados para o diretório destino"""
    print(f"\nCopiando {len(datasets)} datasets para: {diretorio_destino}")
    
    for i, dataset in enumerate(datasets, 1):
        caminho_origem = dataset['caminho']
        caminho_destino = os.path.join(diretorio_destino, dataset['arquivo'])
        
        try:
            shutil.copy2(caminho_origem, caminho_destino)
            print(f"{i:3d}. {dataset['arquivo']} - Score: {dataset['score']:.3f}")
        except Exception as e:
            print(f"Erro ao copiar {dataset['arquivo']}: {e}")

def gerar_relatorio_selecao(datasets, diretorio_destino):
    """Gera relatório detalhado da seleção"""
    caminho_relatorio = os.path.join(diretorio_destino, "relatorio_selecao.csv")
    caminho_sumario = os.path.join(diretorio_destino, "sumario_selecao.txt")
    
    # Salvar CSV detalhado
    df_relatorio = pd.DataFrame(datasets)
    df_relatorio.to_csv(caminho_relatorio, index=False, encoding='utf-8')
    
    # Gerar sumário estatístico
    with open(caminho_sumario, 'w', encoding='utf-8') as f:
        f.write("=" * 80 + "\n")
        f.write("SUMÁRIO DA SELEÇÃO DE DATASETS\n")
        f.write("=" * 80 + "\n\n")
        
        f.write(f"Total de datasets selecionados: {len(datasets)}\n\n")
        
        # Estatísticas por protocolo
        protocolos = {}
        for dataset in datasets:
            proto = dataset['protocolo']
            if proto not in protocolos:
                protocolos[proto] = []
            protocolos[proto].append(dataset)
        
        f.write("DISTRIBUIÇÃO POR PROTOCOLO:\n")
        for proto, data in protocolos.items():
            f.write(f"{proto.upper()}: {len(data)} datasets ({len(data)/len(datasets)*100:.1f}%)\n")
        
        f.write("\nESTATÍSTICAS GERAIS:\n")
        f.write(f"Score médio: {np.mean([d['score'] for d in datasets]):.3f}\n")
        f.write(f"Amostras no intervalo (média): {np.mean([d['amostras_intervalo'] for d in datasets]):.1f}\n")
        f.write(f"Completude média: {np.mean([d['completude'] for d in datasets])*100:.1f}%\n")
        f.write(f"Total de amostras (média): {np.mean([d['total_amostras'] for d in datasets]):.1f}\n")
        f.write(f"Intervalo médio: {np.mean([d['intervalo_dias'] for d in datasets]):.1f} dias\n\n")
        
        f.write("TOP 10 DATASETS:\n")
        f.write("Rank | Arquivo | Score | Amostras | Completude | Protocolo | Link\n")
        f.write("-" * 80 + "\n")
        
        for i, dataset in enumerate(datasets[:10], 1):
            f.write(f"{i:4d} | {dataset['arquivo'][:20]}... | {dataset['score']:.3f} | "
                   f"{dataset['amostras_intervalo']:8d} | {dataset['completude']*100:8.1f}% | "
                   f"{dataset['protocolo']:8s} | {dataset['link']}\n")
    
    print(f"\nRelatórios salvos em:")
    print(f"CSV detalhado: {caminho_relatorio}")
    print(f"Sumário: {caminho_sumario}")

def selecionar_por_protocolo(diretorio_origem, diretorio_destino, top_por_protocolo=50):
    """
    Versão alternativa: seleciona top N de cada protocolo
    """
    # Criar diretório de destino
    Path(diretorio_destino).mkdir(parents=True, exist_ok=True)
    
    # Coletar todos os datasets
    todos_datasets = []
    
    for root, dirs, files in os.walk(diretorio_origem):
        for arquivo in files:
            if arquivo.endswith('.csv'):
                caminho_arquivo = os.path.join(root, arquivo)
                
                try:
                    df = pd.read_csv(caminho_arquivo)
                    if len(df) < 10:
                        continue
                    
                    if 'Data' in df.columns:
                        df['Data'] = pd.to_datetime(df['Data'])
                    
                    total_amostras = len(df)
                    falhas_totais = df['Vazao'].isna().sum()
                    completude = 1 - (falhas_totais / total_amostras)
                    
                    intervalo_info = encontrar_intervalo_mais_longo(df)
                    
                    score = calcular_score(
                        intervalo_info['amostras'],
                        completude,
                        total_amostras
                    )
                    
                    protocolo = extrair_protocolo(arquivo)
                    
                    todos_datasets.append({
                        'caminho': caminho_arquivo,
                        'arquivo': arquivo,
                        'score': score,
                        'amostras_intervalo': intervalo_info['amostras'],
                        'completude': completude,
                        'protocolo': protocolo
                    })
                    
                except Exception as e:
                    print(f"Erro: {arquivo} - {e}")
    
    # Separar por protocolo e selecionar top N de cada
    datasets_por_protocolo = {}
    for dataset in todos_datasets:
        proto = dataset['protocolo']
        if proto not in datasets_por_protocolo:
            datasets_por_protocolo[proto] = []
        datasets_por_protocolo[proto].append(dataset)
    
    # Selecionar top N de cada protocolo
    selecionados = []
    for proto, datasets in datasets_por_protocolo.items():
        ordenados = sorted(datasets, key=lambda x: x['score'], reverse=True)
        selecionados.extend(ordenados[:top_por_protocolo])
    
    # Copiar arquivos selecionados
    copiar_datasets_selecionados(selecionados, diretorio_destino)
    
    return selecionados

# Exemplo de uso

diretorio_origem = "../vazao-2024-2025-agg"
diretorio_destino = "../vazao-escolhidos-2"

# Selecionar os 100 melhores datasets globais
print("Selecionando os 100 melhores datasets...")
# melhores_datasets = selecionar_melhores_datasets(
#     diretorio_origem=diretorio_origem,
#     diretorio_destino=diretorio_destino,
#     top_n=100
# )

# Alternativa: selecionar 100 de cada protocolo
melhores_datasets = selecionar_por_protocolo(
    diretorio_origem=diretorio_origem,
    diretorio_destino=diretorio_destino,
    top_por_protocolo=15
)

print("\n✅ Seleção concluída!")
print(f"📁 {len(melhores_datasets)} datasets copiados para: {diretorio_destino}")